# Enrollment Prediction Model Training - Google Colab

This notebook trains the BioBERT-based enrollment prediction model using Google Colab's free T4 GPU.

**Estimated time: 30-45 minutes on T4 GPU**

## Setup Instructions:
1. **Enable GPU**: Runtime → Change runtime type → Hardware accelerator → GPU (T4)
2. **Set ChromaDB credentials** in the cell below
3. **Run all cells**
4. **Download trained model** at the end

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers chromadb scikit-learn tqdm

## 2. Set ChromaDB Credentials

**⚠️ IMPORTANT: Replace with your actual credentials**

In [ ]:
import os

# Set your ChromaDB credentials here
os.environ['CHROMA_API_KEY'] = 'your_api_key_here'
os.environ['CHROMA_TENANT'] = 'your_tenant_here'
os.environ['CHROMA_DATABASE'] = 'ClinicalAgents'
os.environ['CHROMA_COLLECTION'] = 'clinical_trials'

print("✓ Credentials set")

## 3. Upload Model Files

Upload these files from your local `ml_models/` directory:
- `enrollment_predictor.py`
- `training.py`

In [ ]:
from google.colab import files

print("Upload enrollment_predictor.py and training.py")
uploaded = files.upload()

# Create ml_models directory structure
!mkdir -p ml_models
!mv enrollment_predictor.py ml_models/
!mv training.py ml_models/

# Create __init__.py
with open('ml_models/__init__.py', 'w') as f:
    f.write('from .enrollment_predictor import *\n')
    f.write('from .training import *\n')

print("✓ Files uploaded and organized")

## 4. Verify GPU

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ GPU not detected! Enable GPU in Runtime → Change runtime type")

## 5. Train Model

This will take ~30-45 minutes on T4 GPU

In [ ]:
from ml_models.training import train_enrollment_model

# Train the model
history = train_enrollment_model(
    collection_name='clinical_trials',
    epochs=10,
    batch_size=32,  # Larger batch size for T4 GPU
    learning_rate=2e-5,
    max_samples=None,  # Use all data
    save_dir='saved_models',
    freeze_bert=False
)

print("\n✨ Training complete!")
print(f"Best F1 Score: {max(history['val_f1']):.4f}")

## 6. Download Trained Model

In [ ]:
# Zip the model files
!zip -r enrollment_model.zip saved_models/

# Download
from google.colab import files
files.download('enrollment_model.zip')

print("✓ Model downloaded!")
print("\nExtract this zip file to: agents_server/ml_models/saved_models/")

## 7. Test the Model (Optional)

In [ ]:
# Quick test
from ml_models.enrollment_predictor import EnrollmentFusionModel
from transformers import AutoTokenizer
import torch

# Load model
checkpoint = torch.load('saved_models/enrollment_model.pt')
model = EnrollmentFusionModel()
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✓ Model loaded successfully!")
print(f"Classes: success, delayed, fail")
print(f"Feature mean: {checkpoint['feature_mean']}")
print(f"Feature std: {checkpoint['feature_std']}")